# Aggregating the w2 prediction results
In this notebook, we will aggregate the wasserstein distance results across all the models.

Unlike fate accuracy, the starting cells are all staying in HSC states at the first time point (day2). In this task, the starting cells can be either HSPC or commited progenitors at day4. The starting cells are all held out during training. We then measure their distance to their descendent both in population wise (Well=2) and clone wise (92 clones). 

In [3]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import json

In [4]:
os.chdir("/rds/user/wz369/hpc-work/PINN_dynamics")
higher_dir = '/rds/user/wz369/hpc-work'

# pseudodynamics+ (pdp+)

## PC 30

In [5]:
import re
def w2_file_regex(filename):
    patterns = re.match(r"t(\S{,3})_n(\S{,3})_(\S{0,3})_w2_eval.csv", filename).groups()
    return patterns

In [66]:
eval_dir = f"{higher_dir}/pseudodynamics_plus/results/pseudodynamics+/" 

PCconfig_repeats = [
    "klein_PC_30_lD1_lv1_lgNone",
    "klein_PC30_lD1_cfm1_lgNone",
    "klein_PC30_lD1_cfm1_lv1_lgNone",
    "klein_PC30_lD1_cfm2_lgNone",
    "klein_PC30_lD1_cfm10_lgNone",
    "klein_PC30_lD1_cfm10_lgNone_b1024"
    ]

# loops through all the eval files
results = []
for repeat in PCconfig_repeats:
    eval_subfolder = os.path.join(eval_dir, repeat)
    for file in os.listdir(eval_subfolder):
        if file.endswith("_w2_eval.csv"):
            int_time, noise, sim_fn = w2_file_regex(file)
            # print(int_time, noise, sim_fn)
            df = pd.read_csv(os.path.join(eval_subfolder, file))
            # df['sim_fn'] = sim_fn
            if "w2_scaled" not in df.columns:
                continue
            else:
                results.append(df)
            
# drop the intemediate simulation results
pdp_eval_results = pd.concat(results).query("`noise_scale` == 1.0 & `t_end_norm` > 2.5")

In [67]:
pdp_PC = pdp_eval_results.sort_values(
    by=['w2_raw'], ascending=True
        ).drop_duplicates(subset=['model_dir'], keep='first')
pdp_PC['space'] = 'PC30'
pdp_PC['method'] = 'pseudodynamics+'

In [68]:
PC_fate_pdp = pd.read_csv("scripts/pdp_ranked_perform_PC.csv")
pdp_PC = pdp_PC.merge(PC_fate_pdp,  left_on=['model_dir'], right_on=['model_dir'])

In [69]:
pdp_PC

,w2_scaled,w2_raw,n_sims_x,t_end_norm_x,noise_scale_x,sim_fn_x,n_steps,model_dir,unstandardized,space,...,pearson_r,pearson_p,n_start_cells,n_sims_y,k_nn,obsm_key,n_dims,t_end_norm_y,noise_scale_y,sim_fn_y
0,3.926737,10.628751,10,4.0,1.0,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,...,0.966306,0.000005,2031,100,15,X_pca_scaled,30,2.0,1.0,sb
1,4.052824,10.659013,10,3.0,1.0,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,...,0.949796,0.000026,2031,100,15,X_pca_scaled,30,0.5,2.0,sb
2,4.070421,10.719689,10,3.0,1.0,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,...,0.911329,0.000243,2031,100,15,X_pca_scaled,30,0.5,1.0,sb
3,4.079931,10.801499,10,3.0,1.0,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,...,0.925271,0.000125,2031,100,15,X_pca_scaled,30,0.5,1.5,sb
4,4.034823,10.990043,10,3.5,1.0,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,...,0.962622,0.000008,2031,100,15,X_pca_scaled,30,2.0,1.0,sb
5,4.122865,11.178089,10,3.5,1.0,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,...,0.971981,0.000003,2031,100,15,X_pca_scaled,30,2.0,1.0,sb


In [70]:
pdp_PC.model_dir.str.slice(50,)

0    klein_PC30_lD1_cfm1_lv1_lgNone/pde_params_tsen...
1    klein_PC30_lD1_cfm2_lgNone/pde_params_tsense/V...
2    klein_PC30_lD1_cfm10_lgNone/pde_params_tsense/...
3    klein_PC30_lD1_cfm10_lgNone_b1024/pde_params_t...
4    klein_PC_30_lD1_lv1_lgNone/pde_params_tsense/V...
5    klein_PC30_lD1_cfm1_lgNone/pde_params_tsense/V...
Name: model_dir, dtype: object

## Diffusion map (10 dimension)

In [71]:
eval_dir = f"{higher_dir}/pseudodynamics_plus/results/pseudodynamics+/" 

PCconfig_repeats = [
    "klein_DMscaled_10_cfm5_b512",
    "klein_DMscaled_10_cfm5_b1024",
    "klein_DMscaled_10_cfm10_b512",
    "klein_DMscaled_10_cfm10_b1024",
    ]

# loops through all the eval files
results = []
for repeat in PCconfig_repeats:
    eval_subfolder = os.path.join(eval_dir, repeat)
    for file in os.listdir(eval_subfolder):
        if file.endswith("_w2_eval.csv"):
            int_time, noise, sim_fn = w2_file_regex(file)
            # print(int_time, noise, sim_fn)
            df = pd.read_csv(os.path.join(eval_subfolder, file))
            df['sim_fn'] = sim_fn
            if "w2_scaled" not in df.columns:
                continue
            else:
                results.append(df)
            
# drop the intemediate simulation results
pdp_eval_results = pd.concat(results).query("`t_end_norm`==3.0")
pdp_DM = pdp_eval_results.sort_values(
    by=['w2_raw'], ascending=True
        ).drop_duplicates(subset=['model_dir'], keep='first')
pdp_DM['space'] = 'DM10'
pdp_DM['method'] = 'pseudodynamics+'
pdp_DM

,w2_scaled,w2_raw,n_sims,t_end_norm,noise_scale,sim_fn,n_steps,model_dir,unstandardized,space,method
0,2.655703,0.006539,10,3.0,0.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,DM10,pseudodynamics+
0,2.743906,0.006672,10,3.0,0.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,DM10,pseudodynamics+
0,2.849709,0.006931,10,3.0,0.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,DM10,pseudodynamics+
0,3.089244,0.007420,10,3.0,0.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,DM10,pseudodynamics+


In [72]:
DM_fate_pdp  = pd.read_csv("scripts/pdp_ranked_perform_DM.csv")
pdp_DM = pdp_DM.merge(DM_fate_pdp,  left_on=['model_dir'], right_on=['model_dir'])

# DeepRUOT

In [73]:
eval_dir = f"{higher_dir}/DeepRUOTv2/results/"
print(eval_dir)

/rds/user/wz369/hpc-work/DeepRUOTv2/results/


In [74]:
PC_repeats = [
    'klein_pca30', 'klein_pca30_s0', 'klein_pca30_s10', 'klein_pca30_s42'
]

DeepRUOTresults = []
for repeat in PC_repeats:
    print(repeat)
    pf = pd.read_csv(os.path.join(eval_dir, repeat, "eval_combined.csv"))
    
    DeepRUOTresults.append(pf)

DeepRUOT_pc = pd.concat(DeepRUOTresults) 
DeepRUOT_pc['space'] = 'PC30'
DeepRUOT_pc['method'] = 'DeepRUOTv2'
DeepRUOT_pc

klein_pca30
klein_pca30_s0
klein_pca30_s10
klein_pca30_s42


,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.360906,0.912658,11.860204,11.860204,2031,1,15,PC30,DeepRUOTv2
0,ode,0.334810,0.975341,11.517741,11.517741,2031,1,15,PC30,DeepRUOTv2
0,ode,0.341704,0.822840,11.682735,11.682735,2031,1,15,PC30,DeepRUOTv2
0,ode,0.324471,0.646240,11.315379,11.315379,2031,1,15,PC30,DeepRUOTv2


In [75]:
DMrepeats = [
    'klein_dm10', 'klein_dm10_s0', 'klein_dm10_s10', 'klein_dm10_s42'
]

DeepRUOTresults = []
for repeat in DMrepeats:
    print(repeat)
    pf = pd.read_csv(os.path.join(eval_dir, repeat, "eval_combined.csv"))
    
    DeepRUOTresults.append(pf)

DeepRUOT_dm = pd.concat(DeepRUOTresults) 
DeepRUOT_dm['space'] = 'DM10'
DeepRUOT_dm['method'] = 'DeepRUOTv2'
DeepRUOT_dm

klein_dm10
klein_dm10_s0
klein_dm10_s10
klein_dm10_s42


,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.076809,0.809221,0.006889,0.006889,2031,1,15,DM10,DeepRUOTv2
0,ode,0.164451,0.731430,0.007299,0.007299,2031,1,15,DM10,DeepRUOTv2
0,ode,0.084687,0.815819,0.007517,0.007517,2031,1,15,DM10,DeepRUOTv2
0,ode,0.147710,0.739281,0.006890,0.006890,2031,1,15,DM10,DeepRUOTv2


# scDiffeq

In [76]:
results = []
for seed in [0,1,2]:
    eval_dir = f"{higher_dir}/scDiffEq/results/seeds/seed_{seed}/pca30/plain_sde/fate_prediction_metrics/last"
    df = pd.read_csv(os.path.join(eval_dir, "eval_combined.csv"))
    results.append(df)

scDiffeq_pc = pd.concat(results)
scDiffeq_pc['space'] = 'PC30' 
scDiffeq_pc['method'] = 'scDiffEq'
scDiffeq_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,sde,0.385032,0.846869,10.490688,10.490688,2031,200,15,PC30,scDiffEq
0,sde,0.377646,0.846252,10.662208,10.662208,2031,200,15,PC30,scDiffEq
0,sde,0.373215,0.840821,10.662832,10.662832,2031,200,15,PC30,scDiffEq


In [77]:
results = []
for seed in [0,1,2]:
    eval_dir = f"{higher_dir}/scDiffEq/results/seeds/seed_{seed}/dm10/plain_sde/fate_prediction_metrics/last"
    df = pd.read_csv(os.path.join(eval_dir, "eval_combined.csv"))
    results.append(df)

scDiffeq_dm = pd.concat(results)
scDiffeq_dm['space'] = 'DM10' 
scDiffeq_dm['method'] = 'scDiffEq'
scDiffeq_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,sde,0.434761,0.734353,0.007385,0.007385,2031,200,15,DM10,scDiffEq
0,sde,0.274742,0.908217,0.007252,0.007252,2031,200,15,DM10,scDiffEq
0,sde,0.383555,0.777750,0.007415,0.007415,2031,200,15,DM10,scDiffEq


# TIGON
To run evaluation :

```shell
bash /rds/user/wz369/hpc-work/PINN_dynamics/scripts/TIGON/03_evaluate.sh
```

In [78]:
results = []
for repeat in ["model", "model_seed2", "model_seed3"]:
    df = pd.read_csv(f"{higher_dir}/PINN_dynamics/logs/TIGON/pca/{repeat}/eval_combined.csv")
    results.append(df)
TIGON_pc = pd.concat(results)
TIGON_pc['space'] = 'PC30'
TIGON_pc['method'] = 'TIGON'
TIGON_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.038897,0.980651,4.251537,12.15261,2031,1,15,PC30,TIGON
0,ode,0.038897,0.982976,4.250278,12.14623,2031,1,15,PC30,TIGON
0,ode,0.038897,0.982976,4.250278,12.14623,2031,1,15,PC30,TIGON


In [79]:
results = []
for repeat in ["model", "model_seed2", "model_seed3"]:
    df = pd.read_csv(f"{higher_dir}/PINN_dynamics/logs/TIGON/dm/{repeat}/eval_combined.csv")
    results.append(df)
TIGON_dm = pd.concat(results)
TIGON_dm['space'] = 'DM10'
TIGON_dm['method'] = 'TIGON'
TIGON_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.078779,0.873658,2.837012,0.007061,2031,1,15,DM10,TIGON
0,ode,0.036435,0.963030,2.803302,0.006978,2031,1,15,DM10,TIGON
0,ode,0.036435,0.963030,2.803302,0.006978,2031,1,15,DM10,TIGON


# MIOFlow

In [80]:
results = []
for repeat in [1,2,3]:
    df = pd.read_csv(f"logs/MIOFlow/klein_pca_gaga_run{repeat}/eval_combined.csv")
    # df = pd.read_csv(f"logs/MIOFlow/klein_pca_gaga_latent15_run{repeat}/eval_combined.csv")
    results.append(df)
mioflow_pc = pd.concat(results)
mioflow_pc['space'] = 'PC30'
mioflow_pc['method'] = 'MIOFlow'
mioflow_pc

,sim_mode,accuracy,pearson_r,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.179714,0.954438,13.410164,2031,1,15,PC30,MIOFlow
0,ode,0.157065,0.218102,12.622305,2031,1,15,PC30,MIOFlow
0,ode,0.284097,0.826622,13.920834,2031,1,15,PC30,MIOFlow


In [81]:
results = []
for repeat in [1,2,3]:
    df = pd.read_csv(f"logs/MIOFlow/klein_dm_gaga_run{repeat}/eval_combined.csv")
    # df = pd.read_csv(f"logs/MIOFlow/klein_pca_gaga_latent15_run{repeat}/eval_combined.csv")
    results.append(df)
mioflow_dm = pd.concat(results)
mioflow_dm['space'] = 'DM10'
mioflow_dm['method'] = 'MIOFlow'
mioflow_dm

,sim_mode,accuracy,pearson_r,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.134909,0.187058,0.007028,2031,1,15,DM10,MIOFlow
0,ode,0.270310,0.670564,0.006610,2031,1,15,DM10,MIOFlow
0,ode,0.320039,0.771896,0.006718,2031,1,15,DM10,MIOFlow


# TrajectoryNet

In [82]:
results = []
for repeat in ['model','model_run2','model_run3','model_run4']:
    df = pd.read_csv(f"logs/TrajectoryNet/pca30/{repeat}/eval_combined.csv")
    results.append(df)

TJN_pc = pd.concat(results)
TJN_pc['space'] = 'PC30'
TJN_pc['method'] = 'TrajectoryNet'
TJN_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.092565,0.780078,4.696540,12.383142,2031,1,15,PC30,TrajectoryNet
0,ode,0.084195,0.823549,4.762419,12.415095,2031,1,15,PC30,TrajectoryNet
0,ode,0.117676,0.764594,4.759443,12.445251,2031,1,15,PC30,TrajectoryNet
0,ode,0.092073,0.778220,4.810399,12.521586,2031,1,15,PC30,TrajectoryNet


In [83]:
results = []
for repeat in ['model','model_run2','model_run3']:
    df = pd.read_csv(f"logs/TrajectoryNet/dm10/{repeat}/eval_combined.csv")
    results.append(df)

TJN_dm = pd.concat(results)
TJN_dm['space'] = 'DM10'
TJN_dm['method'] = 'TrajectoryNet'
TJN_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.129493,0.665401,3.010080,0.007514,2031,1,15,DM10,TrajectoryNet
0,ode,0.205810,0.687102,3.066409,0.007632,2031,1,15,DM10,TrajectoryNet
0,ode,0.129000,0.777138,2.866266,0.007135,2031,1,15,DM10,TrajectoryNet


# PRESICENT

In [84]:
results = [] 
for seed in [1,11,22]:
    df = pd.read_csv(f"results/PRESCIENT/pca_scaled_run/PCA_SCALED-softplus_1_500-1e-06/seed_{seed}/eval_combined.csv")
    results.append(df)

PRESCIENT_pc = pd.concat(results)
PRESCIENT_pc['space'] = 'PC30'
PRESCIENT_pc['method'] = 'PRESCIENT'
PRESCIENT_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,sde,0.500739,0.938161,4.175768,12.496020,2031,100,15,PC30,PRESCIENT
0,sde,0.509109,0.928459,4.082661,11.112948,2031,100,15,PC30,PRESCIENT
0,sde,0.510094,0.919101,4.105489,11.209571,2031,100,15,PC30,PRESCIENT


In [85]:
results = [] 
for seed in [0,2,42]:
    df = pd.read_csv(f"results/PRESCIENT/dm_scaled_run/DM_SCALED-softplus_4_64-1e-06/seed_{seed}/eval_combined.csv")
    results.append(df)

PRESCIENT_dm = pd.concat(results)
PRESCIENT_dm['space'] = 'DM10'
PRESCIENT_dm['method'] = 'PRESCIENT'
PRESCIENT_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,sde,0.505662,0.910067,3.641027,0.008753,2031,100,15,DM10,PRESCIENT
0,sde,0.516002,0.938812,3.728569,0.008835,2031,100,15,DM10,PRESCIENT
0,sde,0.533727,0.954560,3.664660,0.008713,2031,100,15,DM10,PRESCIENT


# SF2M 

In [86]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/sf2m/pca30/model_r{r}/eval_combined.csv"))

sf2m_pc = pd.concat(results)
sf2m_pc['space'] = 'PC30'
sf2m_pc['method'] = 'SF2M'
sf2m_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.263909,0.693820,4.014820,10.452632,2031,1,15,PC30,SF2M
1,sde,0.281142,0.703957,3.974596,10.296700,2031,100,15,PC30,SF2M
0,ode,0.263909,0.693820,4.014820,10.452632,2031,1,15,PC30,SF2M
1,sde,0.279173,0.703878,3.975438,10.296235,2031,100,15,PC30,SF2M
0,ode,0.263909,0.693820,4.014820,10.452632,2031,1,15,PC30,SF2M
1,sde,0.284097,0.704065,3.975195,10.299811,2031,100,15,PC30,SF2M


In [87]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/sf2m/dm10_scaled/model_r{r}/eval_combined.csv"))
sf2m_dm = pd.concat(results)
sf2m_dm['space'] = 'DM10'
sf2m_dm['method'] = 'SF2M'
sf2m_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.232890,0.419505,2.560305,0.006292,2031,1,15,DM10,SF2M
1,sde,0.424421,0.698499,2.494664,0.006121,2031,100,15,DM10,SF2M
0,ode,0.451994,0.812459,2.582118,0.006330,2031,1,15,DM10,SF2M
1,sde,0.525357,0.831909,2.526200,0.006191,2031,100,15,DM10,SF2M
0,ode,0.278680,0.371840,2.543430,0.006287,2031,1,15,DM10,SF2M
1,sde,0.435746,0.759428,2.477023,0.006105,2031,100,15,DM10,SF2M


# OTCFM

In [88]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/otcfm/pca30/model_r{r}/eval_combined.csv"))

otcfm_pc = pd.concat(results)
otcfm_pc['space'] = 'PC30'
otcfm_pc['method'] = 'OTCFM'
otcfm_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.312654,0.718214,4.138142,10.914734,2031,1,15,PC30,OTCFM
0,ode,0.312654,0.718214,4.138142,10.914734,2031,1,15,PC30,OTCFM
0,ode,0.312654,0.718214,4.138142,10.914734,2031,1,15,PC30,OTCFM


In [89]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/otcfm/dm10/model_r{r}/eval_combined.csv"))
otcfm_dm = pd.concat(results)
otcfm_dm['space'] = 'DM10'
otcfm_dm['method'] = 'OTCFM'
otcfm_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.372723,0.848451,2.521864,0.006332,2031,1,15,DM10,OTCFM
0,ode,0.372723,0.848451,2.521864,0.006332,2031,1,15,DM10,OTCFM
0,ode,0.372723,0.848451,2.521864,0.006332,2031,1,15,DM10,OTCFM


# Aggregate

In [90]:
pc_df_list = [pdp_PC, DeepRUOT_pc, scDiffeq_pc, TIGON_pc,mioflow_pc,TJN_pc,PRESCIENT_pc,sf2m_pc,otcfm_pc]

# only the neccessary columns
common_col = set(DeepRUOT_pc.columns)
for i in range(len(pc_df_list)):
    common_col = common_col.intersection(set(pc_df_list[i].columns))
common_col = list(common_col)
common_col.remove('n_start_cells')

# aggregating the methods run in the PC space
pc_performs = pd.concat([df.loc[:, common_col] for df in pc_df_list])

# aggregating the methods run in the DM space
dm_df_list = [pdp_DM, DeepRUOT_dm, scDiffeq_dm, TIGON_dm, mioflow_dm, TJN_dm, PRESCIENT_dm, sf2m_dm, otcfm_dm]
dm_performs = pd.concat([df.loc[:, common_col] for df in dm_df_list])

# aggregating the overall
overall_performs = pd.concat([dm_performs, pc_performs], axis=0)

In [91]:
overall_performs.query("`method` == 'PRESCIENT'")

,accuracy,w2_raw,space,method,pearson_r
0,0.505662,0.008753,DM10,PRESCIENT,0.910067
0,0.516002,0.008835,DM10,PRESCIENT,0.938812
0,0.533727,0.008713,DM10,PRESCIENT,0.954560
0,0.500739,12.496020,PC30,PRESCIENT,0.938161
0,0.509109,11.112948,PC30,PRESCIENT,0.928459
0,0.510094,11.209571,PC30,PRESCIENT,0.919101


In [92]:
overall_performs.to_csv("/rds/user/wz369/hpc-work/pseudodynamics_plus/results/benchmarking/klein_overall_performs_May8.csv", index=False)